In [1]:
from ndtools import staged_max_flow as smf

from pathlib import Path
import sys
import json

sys.path.insert(0, str(Path.cwd().parents[2]))
from rsr import rsr
import torch

In [2]:
import importlib
importlib.reload(rsr)

<module 'rsr.rsr' from 'c:\\Users\\bjieu\\git\\rsr\\rsr\\rsr.py'>

# Reliability analysis of benchmark gas supply plant

## Case 1: Original system

### Load data

In [3]:
DATASET = Path(r"data") 

nodes = json.loads((DATASET / "nodes.json").read_text(encoding="utf-8"))
edges = json.loads((DATASET / "edges.json").read_text(encoding="utf-8"))
probs_dict = json.loads((DATASET / "probs.json").read_text(encoding="utf-8"))


In [4]:
def s_fun(comps_st):
    flow, sys_st_str, min_comp_state = smf.sys_fun( comps_st, nodes, edges, probs_dict, target_flow = 0.5 )

    sys_st = 1 if sys_st_str == 's' else 0
    return flow, sys_st, min_comp_state

row_names = list(probs_dict.keys())
n_state = 2  # binary states: 0, 1

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
probs = [[probs_dict[n]['0']['p'], probs_dict[n]['1']['p']] for n in row_names]
probs = torch.tensor(probs, dtype=torch.float32, device=device)

### Get rules for system event

In [5]:
sys_upper_st = 1 

result = rsr.run_ref_extraction_by_mcs(
    # Problem-specific callables / data
    sfun=s_fun,
    probs=probs,
    row_names=row_names,
    n_state=n_state,
    sys_upper_st=sys_upper_st,
    unk_prob_thres = 1e-6,
    output_dir="rsr_res"
) 

---
Round: 1, Unk. prob.: 1.000e+00
Upper probs: 0.000e+00, Lower probs: 0.000e+00
No. of non-dominant refs: 0, Survival refs: 0, Failure refs: 0
Failure sample found from sampling.
No. of existing refs removed:  0
New ref added. System state: 0, System value: -0.0. Total samples: 100000.
New ref (No. of conditions: 1): {'x35': ('<=', 0), 'sys': ('<=', 0)}
Updated sys_vals: [0.0]
---
Round: 2, Unk. prob.: 1.000e+00
Upper probs: 0.000e+00, Lower probs: 0.000e+00
No. of non-dominant refs: 1, Survival refs: 0, Failure refs: 1
Survival sample found from sampling.
No. of existing refs removed:  0
New ref added. System state: 1, System value: 0.5. Total samples: 100000.
New ref (No. of conditions: 17): {'x2': ('>=', 1), 'x3': ('>=', 1), 'x5': ('>=', 1), 'x29': ('>=', 1), 'x30': ('>=', 1), 'x32': ('>=', 1), 'x33': ('>=', 1), 'x34': ('>=', 1), 'x35': ('>=', 1), 'x36': ('>=', 1), 'x38': ('>=', 1), 'x39': ('>=', 1), 'x46': ('>=', 1), 'x61': ('>=', 1), 'x83': ('>=', 1), 'x86': ('>=', 1), 'x87': (

# Modified system

By adding the six most important components, and removing the six least, according to Birbaum's measure.

In [6]:
comps_to_add = ['x30', 'x33', 'x35', 'x46', 'x61', 'x87']
comps_to_remove = ['x22', 'x23', 'x24', 'x25', 'x26', 'x27']

Load data to find rules.

In [7]:
import copy

DATASET = Path(r"data") 

nodes = json.loads((DATASET / "nodes.json").read_text(encoding="utf-8"))
edges = json.loads((DATASET / "edges.json").read_text(encoding="utf-8"))
probs_dict = json.loads((DATASET / "probs.json").read_text(encoding="utf-8"))

new_nodes, new_edges, new_probs =copy.deepcopy(nodes), copy.deepcopy(edges), copy.deepcopy(probs_dict)
for x in comps_to_add:
    new_nodes, new_edges, new_probs = smf.add_a_component(x, new_nodes, new_edges, new_probs)
for x in comps_to_remove:
    new_nodes, new_edges, new_probs = smf.deactivate_a_component(x, new_nodes, new_edges, new_probs)

def s_fun(comps_st):
    flow, sys_st_str, min_comp_state = smf.sys_fun( comps_st, new_nodes, new_edges, new_probs, target_flow = 0.5 )

    sys_st = 1 if sys_st_str == 's' else 0
    return flow, sys_st, min_comp_state

row_names = list(new_probs.keys())
n_state = 2  # binary states: 0, 1
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
probs = [[new_probs[n]['0']['p'], new_probs[n]['1']['p']] for n in row_names]
probs = torch.tensor(probs, dtype=torch.float32, device=device)

In [8]:
sys_upper_st = 1 

result = rsr.run_ref_extraction_by_mcs(
    # Problem-specific callables / data
    sfun=s_fun,
    probs=probs,
    row_names=row_names,
    n_state=n_state,
    sys_upper_st=sys_upper_st,
    unk_prob_thres = 1e-6,
    output_dir="rsr_res_mod"
) 

---
Round: 1, Unk. prob.: 1.000e+00
Upper probs: 0.000e+00, Lower probs: 0.000e+00
No. of non-dominant refs: 0, Survival refs: 0, Failure refs: 0
Survival sample found from sampling.
No. of existing refs removed:  0
New ref added. System state: 1, System value: 0.5. Total samples: 100000.
New ref (No. of conditions: 17): {'x1': ('>=', 1), 'x9': ('>=', 1), 'x19': ('>=', 1), 'x29': ('>=', 1), 'x31': ('>=', 1), 'x33': ('>=', 1), 'x47': ('>=', 1), 'x48': ('>=', 1), 'x68': ('>=', 1), 'x69': ('>=', 1), 'x83': ('>=', 1), 'x86': ('>=', 1), 'x87': ('>=', 1), 'x30_copy': ('>=', 1), 'x35_copy': ('>=', 1), 'x46_copy': ('>=', 1), 'x61_copy': ('>=', 1), 'sys': ('>=', 1)}
Updated sys_vals: [0.5]
---
Round: 2, Unk. prob.: 1.000e+00
Upper probs: 0.000e+00, Lower probs: 0.000e+00
No. of non-dominant refs: 1, Survival refs: 1, Failure refs: 0
Failure sample found from sampling.
No. of existing refs removed:  0
New ref added. System state: 0, System value: -0.0. Total samples: 100000.
New ref (No. of cond